# HOG size and sex bias

Load packages

In [207]:
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

Add function for Wilson confidence interval, instead of traditional CI, as the proportions for some sizes are 0 or 1. 

In [208]:
def wilson_ci(k, n, z=1.96):
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    margin = z * np.sqrt((p*(1-p) + z**2/(4*n)) / n) / denom
    return center - margin, center + margin

#z = z-score corresponding here to 1.96 for 95% interval.  
#p = p(hat) = proportion of success. 
#k = sucesses 
#n = number of trials 

Load the annotated result dataset from Salmon map

In [209]:
salmon_map_full_annot = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/salmon_map_dominance_DE_sex_results_new_filtering_genotype_controlled_age_rank.csv", float_precision='legacy')

# Replace the zeros in padj with 1e-308 avoid log10 issues
salmon_map_full_annot["padj_safe"] = salmon_map_full_annot["padj"].replace(0, 1e-308).fillna(1)

# Add the negative log 10 padj for plotting
salmon_map_full_annot["neglog10_padj"] = -np.log10(salmon_map_full_annot["padj_safe"])

# Add significance to differentially expressed genes 
salmon_map_full_annot["significant"] = (
    (salmon_map_full_annot["padj"] < 0.05) &
    (salmon_map_full_annot["log2FoldChange"].abs() > 1)
)

# Add a label to the significant genes
salmon_map_full_annot["label"] = salmon_map_full_annot["gene_id"].where(salmon_map_full_annot["significant"], "")

salmon_map_full_annot

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,duplication_node,gene_tree_node,duplication_support,duplication_type,node_depth_from_root,branch_length,padj_safe,neglog10_padj,significant,label
0,248.420188,0.640709,0.110526,5.796904,6.755020e-09,1.316673e-08,g2.t1,g2,utg000001l,220384.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n4,1.0,Terminal,0.736662,0.052941,1.316673e-08,7.880522,False,
1,92.726112,0.650988,0.104454,6.232321,4.595741e-10,9.353175e-10,g3.t1,g3,utg000001l,227675.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n15,1.0,Terminal,0.736662,0.052941,9.353175e-10,9.029041,False,
2,136.044931,-0.600974,0.070022,-8.582668,9.269779e-18,2.387530e-17,g4.t1,g4,utg000001l,245866.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n14,1.0,Terminal,0.736662,0.052941,2.387530e-17,16.622051,False,
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,257697.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n13,1.0,Terminal,0.736662,0.052941,2.671748e-04,3.573204,True,g5
4,381.084979,-0.875080,0.046175,-18.951587,4.284679e-80,3.659726e-79,g6.t1,g6,utg000001l,263441.0,...,NaN,NaN,NaN,NaN,NaN,NaN,3.659726e-79,78.436551,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17569,20.275098,0.319252,0.150056,2.127555,3.337403e-02,4.238130e-02,g34843.t1,g34843,utg003648l,5656.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n6,1.0,Terminal,0.736662,0.052941,4.238130e-02,1.372826,False,
17570,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,19710.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n12,1.0,Terminal,0.736662,0.052941,6.956768e-04,3.157592,True,g34884
17571,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,1.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n50,1.0,Terminal,0.736662,0.052941,4.718367e-08,7.326208,True,g34922
17572,19.121063,2.284212,0.326813,6.989347,2.761695e-12,6.049015e-12,g35167.t1,g35167,utg003885l,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,6.049015e-12,11.218315,True,g35167


Remove entries that does not have a HOG

In [210]:
salmon_map_results_HOG = salmon_map_full_annot.dropna(subset=["HOG"])
salmon_map_results_HOG
# Down from 17.574 to 15.648 transcripts


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,duplication_node,gene_tree_node,duplication_support,duplication_type,node_depth_from_root,branch_length,padj_safe,neglog10_padj,significant,label
0,248.420188,0.640709,0.110526,5.796904,6.755020e-09,1.316673e-08,g2.t1,g2,utg000001l,220384.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n4,1.0,Terminal,0.736662,0.052941,1.316673e-08,7.880522,False,
1,92.726112,0.650988,0.104454,6.232321,4.595741e-10,9.353175e-10,g3.t1,g3,utg000001l,227675.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n15,1.0,Terminal,0.736662,0.052941,9.353175e-10,9.029041,False,
2,136.044931,-0.600974,0.070022,-8.582668,9.269779e-18,2.387530e-17,g4.t1,g4,utg000001l,245866.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n14,1.0,Terminal,0.736662,0.052941,2.387530e-17,16.622051,False,
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,257697.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n13,1.0,Terminal,0.736662,0.052941,2.671748e-04,3.573204,True,g5
4,381.084979,-0.875080,0.046175,-18.951587,4.284679e-80,3.659726e-79,g6.t1,g6,utg000001l,263441.0,...,NaN,NaN,NaN,NaN,NaN,NaN,3.659726e-79,78.436551,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17567,10.439363,0.527491,0.759882,0.694175,4.875723e-01,5.260053e-01,g34647.t1,g34647,utg003542l,22333.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n2,1.0,Terminal,0.736662,0.052941,5.260053e-01,0.279010,False,
17569,20.275098,0.319252,0.150056,2.127555,3.337403e-02,4.238130e-02,g34843.t1,g34843,utg003648l,5656.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n6,1.0,Terminal,0.736662,0.052941,4.238130e-02,1.372826,False,
17570,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,19710.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n12,1.0,Terminal,0.736662,0.052941,6.956768e-04,3.157592,True,g34884
17571,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,1.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n50,1.0,Terminal,0.736662,0.052941,4.718367e-08,7.326208,True,g34922


Compute number of paralogs in each HOG.  
Add the hog_size column to the results table by merging on HOG name.  

In [211]:
hog_size = (
    salmon_map_results_HOG
    .groupby("HOG")
    .size()
    .reset_index(name="HOG_size")
)

salmon_map_results_HOG = salmon_map_results_HOG.merge(
    hog_size,
    on="HOG",
    how="left"
)

hog_size

hog_size_table = (
    hog_size
    .groupby("HOG_size")
    .size()
    .reset_index(name="n_HOGs")
)

hog_size_table["n_transcripts"] = hog_size_table["HOG_size"] * hog_size_table["n_HOGs"]
hog_size_table

,HOG_size,n_HOGs,n_transcripts
0,1,8173,8173
1,2,1863,3726
2,3,398,1194
3,4,168,672
4,5,83,415
5,6,51,306
6,7,26,182
7,8,30,240
8,9,18,162
9,10,6,60


Add sex bias column

In [212]:
salmon_map_results_HOG["sex_bias"] = "not_significant"

salmon_map_results_HOG.loc[
    salmon_map_results_HOG["significant"] &
    (salmon_map_results_HOG["log2FoldChange"] > 0),
    "sex_bias"
] = "male"

salmon_map_results_HOG.loc[
    salmon_map_results_HOG["significant"] &
    (salmon_map_results_HOG["log2FoldChange"] < 0),
    "sex_bias"
] = "female"
salmon_map_results_HOG

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,duplication_support,duplication_type,node_depth_from_root,branch_length,padj_safe,neglog10_padj,significant,label,HOG_size,sex_bias
0,248.420188,0.640709,0.110526,5.796904,6.755020e-09,1.316673e-08,g2.t1,g2,utg000001l,220384.0,...,1.0,Terminal,0.736662,0.052941,1.316673e-08,7.880522,False,,2,not_significant
1,92.726112,0.650988,0.104454,6.232321,4.595741e-10,9.353175e-10,g3.t1,g3,utg000001l,227675.0,...,1.0,Terminal,0.736662,0.052941,9.353175e-10,9.029041,False,,2,not_significant
2,136.044931,-0.600974,0.070022,-8.582668,9.269779e-18,2.387530e-17,g4.t1,g4,utg000001l,245866.0,...,1.0,Terminal,0.736662,0.052941,2.387530e-17,16.622051,False,,2,not_significant
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,257697.0,...,1.0,Terminal,0.736662,0.052941,2.671748e-04,3.573204,True,g5,2,male
4,381.084979,-0.875080,0.046175,-18.951587,4.284679e-80,3.659726e-79,g6.t1,g6,utg000001l,263441.0,...,NaN,NaN,NaN,NaN,3.659726e-79,78.436551,False,,1,not_significant
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15643,10.439363,0.527491,0.759882,0.694175,4.875723e-01,5.260053e-01,g34647.t1,g34647,utg003542l,22333.0,...,1.0,Terminal,0.736662,0.052941,5.260053e-01,0.279010,False,,8,not_significant
15644,20.275098,0.319252,0.150056,2.127555,3.337403e-02,4.238130e-02,g34843.t1,g34843,utg003648l,5656.0,...,1.0,Terminal,0.736662,0.052941,4.238130e-02,1.372826,False,,3,not_significant
15645,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,19710.0,...,1.0,Terminal,0.736662,0.052941,6.956768e-04,3.157592,True,g34884,8,male
15646,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,1.0,...,1.0,Terminal,0.736662,0.052941,4.718367e-08,7.326208,True,g34922,8,male


# Analysis 1: Bias direction among biased transcripts. 
Which fraction is male within each HOG size? 

Which transcripts that belong to a HOG is biased? Remove the unbiased transcripts

In [213]:
biased = salmon_map_results_HOG[
    salmon_map_results_HOG["sex_bias"].isin(["male", "female"])
].copy()

biased
# 6472 belong to a HOG and are significantly sex biased

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,duplication_support,duplication_type,node_depth_from_root,branch_length,padj_safe,neglog10_padj,significant,label,HOG_size,sex_bias
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,257697.0,...,1.000000,Terminal,0.736662,0.052941,2.671748e-04,3.573204,True,g5,2,male
5,70.718212,-1.039030,0.089153,-11.654489,2.176813e-31,7.787918e-31,g7.t1,g7,utg000001l,371755.0,...,NaN,NaN,NaN,NaN,7.787918e-31,30.108579,True,g7,1,female
6,475.721002,1.120124,0.059428,18.848397,3.028815e-79,2.541218e-78,g8.t1,g8,utg000001l,510966.0,...,NaN,NaN,NaN,NaN,2.541218e-78,77.594958,True,g8,1,male
7,97.175715,1.341896,0.124766,10.755269,5.597042e-27,1.811622e-26,g9.t1,g9,utg000001l,531498.0,...,1.000000,Non-Terminal,0.683721,0.022679,1.811622e-26,25.741932,True,g9,2,male
11,237.620281,-4.015786,0.186764,-21.501956,1.492615e-102,1.902623e-101,g13.t1,g13,utg000001l,666271.0,...,0.333333,Non-Terminal,0.661042,0.031000,1.902623e-101,100.720647,True,g13,1,female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15639,3.429476,2.064888,0.534763,3.861312,1.127796e-04,1.773450e-04,g34258.t1,g34258,utg003361l,24503.0,...,1.000000,Terminal,0.736662,0.052941,1.773450e-04,3.751181,True,g34258,3,male
15642,3.024122,2.592433,0.660156,3.927001,8.601149e-05,1.363904e-04,g34611.t1,g34611,utg003498l,55900.0,...,NaN,NaN,NaN,NaN,1.363904e-04,3.865216,True,g34611,1,male
15645,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,19710.0,...,1.000000,Terminal,0.736662,0.052941,6.956768e-04,3.157592,True,g34884,8,male
15646,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,1.0,...,1.000000,Terminal,0.736662,0.052941,4.718367e-08,7.326208,True,g34922,8,male


Count male and female biased transcripts per HOG size.  
Filter out HOG size = 1 as those have no paralogs

In [214]:
hog_counts = (
    biased
    .groupby(["HOG_size", "sex_bias"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

hog_counts

sex_bias,HOG_size,female,male
0,1,1160,1884
1,2,490,1061
2,3,139,427
3,4,70,278
4,5,28,185
5,6,24,120
6,7,33,66
7,8,23,109
8,9,23,60
9,10,2,24


In [215]:
hog_counts = hog_counts[hog_counts["HOG_size"] >= 2].copy()
hog_counts

sex_bias,HOG_size,female,male
1,2,490,1061
2,3,139,427
3,4,70,278
4,5,28,185
5,6,24,120
6,7,33,66
7,8,23,109
8,9,23,60
9,10,2,24
10,11,8,31


Im doing this as a binomial proportion of p_male = male/(male+female) as some sizes have 0 female biased transcripts.  
Confidence intervals calulated with Wilson score interval, performs better when proportions are close to 0 or 1. 

In [216]:
hog_counts["total_biased"] = hog_counts["male"] + hog_counts["female"]

hog_counts["p_male"] = (
    hog_counts["male"] /
    hog_counts["total_biased"]
)

ci_bounds = hog_counts.apply(
    lambda row: wilson_ci(row["male"], row["total_biased"]),
    axis=1
)

hog_counts["ci_lower"] = [c[0] for c in ci_bounds]
hog_counts["ci_upper"] = [c[1] for c in ci_bounds]

#clipping to avoid floating point residues 
hog_counts["ci_lower"] = hog_counts["ci_lower"].clip(lower=0)
hog_counts["ci_upper"] = hog_counts["ci_upper"].clip(upper=1)


#old normal confidence interval
#hog_counts["se"] = np.sqrt(
#    hog_counts["p_male"] *
#    (1 - hog_counts["p_male"]) /
#    hog_counts["total_biased"]
#)

#hog_counts["ci_lower"] = hog_counts["p_male"] - 1.96 * hog_counts["se"]
#hog_counts["ci_upper"] = hog_counts["p_male"] + 1.96 * hog_counts["se"]
hog_counts


sex_bias,HOG_size,female,male,total_biased,p_male,ci_lower,ci_upper
1,2,490,1061,1551,0.684075,0.660508,0.706732
2,3,139,427,566,0.754417,0.717319,0.788085
3,4,70,278,348,0.798851,0.753574,0.837601
4,5,28,185,213,0.868545,0.816569,0.907462
5,6,24,120,144,0.833333,0.763976,0.885368
6,7,33,66,99,0.666667,0.569119,0.751763
7,8,23,109,132,0.825758,0.752095,0.880996
8,9,23,60,83,0.722892,0.618381,0.807682
9,10,2,24,26,0.923077,0.758581,0.978645
10,11,8,31,39,0.794872,0.644657,0.892204


Plot

In [217]:
fig1 = px.line(
    hog_counts,
    x="HOG_size",
    y="p_male",
    markers=True,
    hover_data={
        "male": True,
        "female": True,
        "total_biased": True,
        "p_male": ":.3f"
    }
)

fig1.update_traces(
    error_y=dict(
        type="data",
        symmetric=False,
        array=hog_counts["ci_upper"] - hog_counts["p_male"],
        arrayminus=hog_counts["p_male"] - hog_counts["ci_lower"]
    )
)

fig1.add_hline(
    y=0.5,
    line_dash="dash",
    line_color="black"
)

fig1.update_layout(
    xaxis_title="HOG size (≥2)",
    yaxis_title="Proportion male biased (p_male = male/male+female)",
)

fig1.show()

# Analysis 2: Male/Female/Unbiased prop within HOG size.  

In [218]:
#number in each category
hog_bias = (
    salmon_map_results_HOG
    .groupby(["HOG_size", "sex_bias"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

#add the total
hog_bias["total"] = (
    hog_bias["male"] +
    hog_bias["female"] +
    hog_bias["not_significant"]
)


hog_bias

sex_bias,HOG_size,female,male,not_significant,total
0,1,1160,1884,5129,8173
1,2,490,1061,2175,3726
2,3,139,427,628,1194
3,4,70,278,324,672
4,5,28,185,202,415
5,6,24,120,162,306
6,7,33,66,83,182
7,8,23,109,108,240
8,9,23,60,79,162
9,10,2,24,34,60


Filter out HOG size 1

In [219]:
hog_bias = hog_bias[hog_bias["HOG_size"] >= 2].copy()
hog_bias

sex_bias,HOG_size,female,male,not_significant,total
1,2,490,1061,2175,3726
2,3,139,427,628,1194
3,4,70,278,324,672
4,5,28,185,202,415
5,6,24,120,162,306
6,7,33,66,83,182
7,8,23,109,108,240
8,9,23,60,79,162
9,10,2,24,34,60
10,11,8,31,27,66


Convert to long format instead of wide format

In [220]:
hog_bias_long = hog_bias.melt(
    id_vars=["HOG_size", "total"],
    value_vars=["male", "female", "not_significant"],
    var_name="sex_bias",
    value_name="transcript_count"
)
hog_bias_long

,HOG_size,total,sex_bias,transcript_count
0,2,3726,male,1061
1,3,1194,male,427
2,4,672,male,278
3,5,415,male,185
4,6,306,male,120
...,...,...,...,...
64,20,40,not_significant,21
65,22,22,not_significant,14
66,23,23,not_significant,14
67,24,24,not_significant,15


Add the proprtions and confidence intervals.  
Confidence intervals calulated with Wilson score interval, performs better when proportions are close to 0 or 1. 

In [221]:
hog_bias_long["proportion"] = (
    hog_bias_long["transcript_count"] /
    hog_bias_long["total"]
)

ci_bounds = hog_bias_long.apply(
    lambda row: wilson_ci(row["transcript_count"], row["total"]),
    axis=1
)

hog_bias_long["ci_lower"] = [c[0] for c in ci_bounds]
hog_bias_long["ci_upper"] = [c[1] for c in ci_bounds]

#clipping to avoid floating point residues 
hog_bias_long["ci_lower"] = hog_bias_long["ci_lower"].clip(lower=0)
hog_bias_long["ci_upper"] = hog_bias_long["ci_upper"].clip(upper=1)


#old normal confidence interval
#hog_bias_long["se"] = np.sqrt(
#    hog_bias_long["proportion"] *
#    (1 - hog_bias_long["proportion"]) /
#    hog_bias_long["total"]
#)

#hog_bias_long["ci_lower"] = hog_bias_long["proportion"] - 1.96 * hog_bias_long["se"]
#hog_bias_long["ci_upper"] = hog_bias_long["proportion"] + 1.96 * hog_bias_long["se"]
hog_bias_long

,HOG_size,total,sex_bias,transcript_count,proportion,ci_lower,ci_upper
0,2,3726,male,1061,0.284756,0.270492,0.299463
1,3,1194,male,427,0.357621,0.330931,0.385225
2,4,672,male,278,0.413690,0.377047,0.451315
3,5,415,male,185,0.445783,0.398675,0.493886
4,6,306,male,120,0.392157,0.339113,0.447874
...,...,...,...,...,...,...,...
64,20,40,not_significant,21,0.525000,0.374971,0.670648
65,22,22,not_significant,14,0.636364,0.429513,0.802670
66,23,23,not_significant,14,0.608696,0.407852,0.778426
67,24,24,not_significant,15,0.625000,0.427096,0.788409


Plot with plotly

In [222]:
fig2 = px.line(
    hog_bias_long,
    x="HOG_size",
    y="proportion",
    color="sex_bias",
    markers=True,
        hover_data={
        "transcript_count": True,
        "total": True,
        "proportion": ":.3f"
    }
)

for trace in fig2.data:
    bias_type = trace.name
    
    subset = hog_bias_long[hog_bias_long["sex_bias"] == bias_type]
    
    trace.error_y = dict(
        type="data",
        symmetric=False,
        array=subset["ci_upper"].values - subset["proportion"].values,
        arrayminus=subset["proportion"].values - subset["ci_lower"].values
    )

fig2.update_layout(
    xaxis_title="HOG size (≥2)",
    yaxis_title="Proportion of transcripts",
)
fig2.show()

# Analysis 3: Variance vs HOG Size
How variable are the log2FoldChange values inside each HOG size? 

Transcript level log2FC variance per HOG size

In [223]:
variance_by_size = (
    salmon_map_results_HOG
    .groupby("HOG_size")["log2FoldChange"]
    .agg(
        variance="var",
        n="count"
    )
    .reset_index()
)

#filter out size 1
variance_by_size = variance_by_size[
    variance_by_size["HOG_size"] >= 2
].copy()

#filter out sizes with 0 transcripts
variance_by_size = variance_by_size[
    variance_by_size["n"] > 0
]


variance_by_size

,HOG_size,variance,n
1,2,6.847107,3726
2,3,8.971673,1194
3,4,8.744715,672
4,5,5.452064,415
5,6,4.033671,306
6,7,7.130890,182
7,8,7.942246,240
8,9,5.910972,162
9,10,3.242385,60
10,11,8.226703,66


Plot

In [224]:
fig3 = px.line(
    variance_by_size,
    x="HOG_size",
    y="variance",
    markers=True,
    hover_data={
        "n": True,
        "variance": ":.3f"
    }
)

fig3.update_layout(
    xaxis_title="HOG size (≥2)",
    yaxis_title="Variance of log2FoldChange",
)

fig3.show()


Expression differene decreases as the family size increases. But for HOG sizes larger than 15 we have very small sample sizes so variance is noisy and unreliable as small sample makes variance unstable

# Analysis 4: Variance within each HOG

In [225]:
variance_within_hog = (
    salmon_map_results_HOG
    .groupby(["HOG", "HOG_size"])["log2FoldChange"]
    .agg(
        variance="var"
    )
    .reset_index()
)

variance_within_hog = variance_within_hog[
    variance_within_hog["HOG_size"] >= 2
]

variance_within_hog

,HOG,HOG_size,variance
0,N0.HOG0000007,17,0.998066
1,N0.HOG0000008,17,1.452074
2,N0.HOG0000009,14,2.942610
3,N0.HOG0000010,25,0.324398
4,N0.HOG0000011,4,0.056720
...,...,...,...
10830,N0.HOG0024502,2,0.081273
10840,N0.HOG0024534,2,0.037674
10842,N0.HOG0024539,2,0.740688
10844,N0.HOG0024541,2,1.677436


Plot

In [226]:
#separate the outlier 
low = variance_within_hog[variance_within_hog["variance"] <= 120]
high = variance_within_hog[variance_within_hog["variance"] > 120]

#create two subplots 
fig4 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.1, 0.9],  # small top, big bottom
    vertical_spacing=0.05
)

#Add bottom panel (main plot)
fig4.add_trace(
    go.Scatter(
        x=low["HOG_size"],
        y=low["variance"],
        mode="markers",
        hovertext=low["HOG"],
        name="Variance"
    ),
    row=2, col=1
)

#Add top panel (outlier)
fig4.add_trace(
    go.Scatter(
        x=high["HOG_size"],
        y=high["variance"],
        mode="markers",
        hovertext=high["HOG"],
        name="Outlier"
    ),
    row=1, col=1
)

#set axis rate
fig4.update_yaxes(range=[0,120], row=2, col=1)
fig4.update_yaxes(range=[260,280], 
                  tickmode="array",
                  tickvals=[260,280],
                  row=1, col=1)

fig4.update_layout(
    height=600,
    showlegend=False,
)
# Set bottom axis titles
fig4.update_xaxes(title_text="HOG size", row=2, col=1, tickmode="linear", dtick=1)
fig4.update_yaxes(title_text="Variance", row=2, col=1)

#top x axis ticks
fig4.update_xaxes(row=1, col=1, tickmode="linear", dtick=1)
fig4.show()

"estimate expression in each sex"  
pick a transcript sequence and paste in conserved domain search (ncbi) to see if it has a TE sequence

# Analysis 5: Within HOG directional bias 

Put the HOGs into categories (biased_sets)

In [227]:
hog_direction = (
    salmon_map_results_HOG
    .groupby(["HOG", "HOG_size"])["sex_bias"]
    .apply(lambda x: set(x))
    .reset_index()
)

hog_direction.rename(columns={"sex_bias": "bias_set"}, inplace=True)

#classify the sets
def classify_bias(bias_set):
    
    if bias_set == {"male"}:
        return "All male biased"
    
    elif bias_set == {"female"}:
        return "All female biased"
    
    elif bias_set == {"not_significant"}:
        return "All unbiased"
    
    elif bias_set == {"male", "female"}:
        return "Male + Female"
    
    elif bias_set == {"male", "not_significant"}:
        return "Male + Unbiased"
    
    elif bias_set == {"female", "not_significant"}:
        return "Female + Unbiased"
    
    elif bias_set == {"male", "female", "not_significant"}:
        return "All three"
    
    else:
        return "Other"
    
hog_direction["category"] = hog_direction["bias_set"].apply(classify_bias)
hog_direction = hog_direction[
    hog_direction["HOG_size"] >= 2
]
hog_direction

,HOG,HOG_size,bias_set,category
0,N0.HOG0000007,17,"{not_significant, male}",Male + Unbiased
1,N0.HOG0000008,17,"{not_significant, male}",Male + Unbiased
2,N0.HOG0000009,14,"{not_significant, male}",Male + Unbiased
3,N0.HOG0000010,25,"{not_significant, male}",Male + Unbiased
4,N0.HOG0000011,4,{female},All female biased
...,...,...,...,...
10830,N0.HOG0024502,2,{male},All male biased
10840,N0.HOG0024534,2,{not_significant},All unbiased
10842,N0.HOG0024539,2,"{not_significant, male}",Male + Unbiased
10844,N0.HOG0024541,2,"{female, not_significant}",Female + Unbiased


Count categories

In [228]:
category_order = [
    "All three",
    "All male biased",
    "Male + Unbiased",
    "Male + Female",
    "Female + Unbiased",
    "All female biased",
    "All unbiased"
]

category_colors = {
    "All three":        "#807c7c",
    "All male biased":  "#0030FF",
    "Male + Unbiased":  "#00D4CA",
    "Male + Female":    "#C400EA",   
    "Female + Unbiased":"#FFD500",
    "All female biased":"#FF0000",
    "All unbiased":     "#00CC51",
}

hog_direction["category"] = pd.Categorical(
    hog_direction["category"],
    categories=category_order,
    ordered=True
)

hog_category_counts = (
    hog_direction
    .groupby("category", observed=False)
    .size()
    .reset_index(name="count")
)

hog_category_counts

,category,count
0,All three,54
1,All male biased,537
2,Male + Unbiased,539
3,Male + Female,37
4,Female + Unbiased,192
5,All female biased,206
6,All unbiased,1112


Plot

In [229]:
fig5 = px.bar(
    hog_category_counts,
    x="category",
    y="count",
    color="category",
    color_discrete_map=category_colors,
    category_orders={"category": category_order}
)

fig5.update_layout(
    xaxis_title="HOG composition category",
    yaxis_title="Number of HOGs",
    xaxis_tickangle=45,
    showlegend=False
)

fig5.show()

Plotly has an issue with the box plots, so here i use go.Figure() to force plotly  

In [230]:
fig6 = go.Figure()

cat_positions = {cat: i for i, cat in enumerate(category_order)}
np.random.seed(42)

for cat in category_order:
    subset = hog_direction[hog_direction["category"] == cat]
    if subset.empty:
        continue

    color  = category_colors[cat]
    x_pos  = cat_positions[cat]

    # Box (no built-in points — we draw them ourselves below)
    fig6.add_trace(go.Box(
        y=subset["HOG_size"],
        x=[x_pos] * len(subset),
        name=cat,
        marker_color=color,
        line_color=color,
        fillcolor=color,
        opacity=0.6,
        boxpoints=False,
        width=0.5,
        showlegend=False
    ))

    # Regular points (size > 2) — circles
    reg = subset[subset["HOG_size"] != 2]
    if not reg.empty:
        fig6.add_trace(go.Scatter(
            x=x_pos + np.random.uniform(-0.15, 0.15, len(reg)),
            y=reg["HOG_size"],
            mode="markers",
            marker=dict(color=color, symbol="circle", size=5, opacity=0.7,
                        line=dict(width=0.5, color="white")),
            showlegend=False,
            text=reg["HOG"].astype(str),
            hovertemplate="HOG: %{text}<br>Size: %{y}<extra></extra>"
        ))

    # HOG size 2 shape — diamond (All three cannot exist at size 2)
    s2 = subset[subset["HOG_size"] == 2]
    if not s2.empty:
        fig6.add_trace(go.Scatter(
            x=x_pos + np.random.uniform(-0.15, 0.15, len(s2)),
            y=s2["HOG_size"],
            mode="markers",
            marker=dict(color=color, symbol="diamond", size=8, opacity=0.9,
                        line=dict(width=0.8, color="white")),
            showlegend=False,
            text=s2["HOG"].astype(str),
            hovertemplate="HOG: %{text}<br>Size: 2 ◆<extra></extra>"
        ))

fig6.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=list(cat_positions.values()),
        ticktext=list(cat_positions.keys()),
        tickangle=45,
        title="HOG composition category"
    ),
    yaxis_title="HOG size",
    showlegend=False
)

fig6.show()

In [231]:
fig6 = px.box(
    hog_direction, 
    x="category", 
    y="HOG_size", 
    color_discrete_map=category_colors,
    category_orders={"category": category_order},
    points="all", 
    hover_data="HOG" 
    ) 

fig6.update_layout( 
    xaxis_title="HOG composition category", 
    yaxis_title="HOG size", 
    xaxis_tickangle=45,
    showlegend=False
    ) 
fig6.show()

In [232]:
fig7 = px.histogram(
    hog_direction,
    x="HOG_size",
    color="category",
    color_discrete_map=category_colors,
    category_orders={"category": category_order},
    barmode="group"
)
fig7.update_xaxes(
    tickmode="linear",
    dtick=1
)
fig7.update_yaxes(
    type="log",
    tickmode="array",
    tickvals=[1, 10, 100, 1000],
    ticktext=["1", "10", "100", "1000"]
)

fig7.update_layout(
    xaxis_title="HOG size",
    yaxis_title="Number of HOGs (log10 scale)"
)

fig7.show()

y axis log scaled. line plot for each category instead. 

Same but line plot

In [233]:
import pandas as pd

# Get full range of sizes
all_sizes = range(hog_direction["HOG_size"].min(),
                  hog_direction["HOG_size"].max() + 1)

# Create full grid
full_index = pd.MultiIndex.from_product(
    [all_sizes, category_order],
    names=["HOG_size", "category"]
)

hog_size_counts = (
    hog_direction
    .groupby(["HOG_size", "category"], observed=False)
    .size()
    .reindex(full_index, fill_value=0)
    .reset_index(name="count")
)

#with log scale we cannot see 0. 
hog_size_counts["count_plot"] = hog_size_counts["count"].replace(0, 0.1)

fig9 = px.line(
    hog_size_counts,
    x="HOG_size",
    y="count_plot",
    color="category",
    color_discrete_map=category_colors,
    category_orders={"category": category_order},
    markers=True,
)

fig9.update_xaxes(
    tickmode="linear",
    dtick=1
)

fig9.update_yaxes(
    type="log",
    tickmode="array",
    tickvals=[1, 10, 100, 1000],
    ticktext=["1", "10", "100", "1000"]
)

fig9.update_layout(
    xaxis_title="HOG size",
    yaxis_title="Number of HOGs (log scale)"
)

fig9.show()

% proportion bar chart

In [234]:
# ── Fig 8 – 100% stacked proportional bar chart (simplified with px.bar) ──

hog_size_cat = (
    hog_direction
    .groupby(["HOG_size", "category"], observed=False)
    .size()
    .reset_index(name="count")
)

size_totals = (
    hog_size_cat.groupby("HOG_size")["count"]
    .sum()
    .reset_index(name="total")
)

hog_size_cat = hog_size_cat.merge(size_totals, on="HOG_size")
hog_size_cat["pct"] = hog_size_cat["count"] / hog_size_cat["total"] * 100
hog_size_cat["label"] = hog_size_cat["count"].where(hog_size_cat["pct"] >= 4, other="")

fig8 = px.bar(
    hog_size_cat,
    x="HOG_size",
    y="pct",
    color="category",
    color_discrete_map=category_colors,
    category_orders={"category": category_order},
    text="label",
    custom_data=["count", "total", "category"],
    barmode="stack"
)

fig8.update_traces(
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(color="white", size=10),
    hovertemplate=(
        "<b>%{customdata[2]}</b><br>"
        "HOG size: %{x}<br>"
        "Proportion: %{y:.1f}%<br>"
        "Count: %{customdata[0]}<br>"
        "Total at this size: %{customdata[1]}"
        "<extra></extra>"
    )
)

# Total n= annotations below each bar
annotations = [
    dict(
        x=row.HOG_size, y=101,
        xref="x", yref="y",
        text=f"n={row.total}",
        showarrow=False,
        font=dict(size=11, color="#444"),
        yanchor="bottom",
        textangle=-45
    )
    for row in size_totals.itertuples()
]

fig8.update_layout(
    xaxis=dict(tickmode="linear", dtick=1, title="HOG size"),
    yaxis=dict(title="Proportion of HOGs (%)", range=[0, 111], ticksuffix="%"),
    legend_title_text="Category",
    annotations=annotations,
    margin=dict(b=60),
    width=1100,
    height=600
)

fig8.show()

Density curve plot

In [235]:
from scipy.stats import gaussian_kde

sizes = hog_direction["HOG_size"]
x_range = np.linspace(sizes.min(), sizes.max(), 300)

kde = gaussian_kde(sizes)
y_vals = kde(x_range) * len(sizes)

fig10 = go.Figure(go.Scatter(
    x=x_range,
    y=y_vals,
    mode="lines",
    fill="tozeroy",
    line=dict(color="#0030FF", width=2),
    fillcolor="rgba(0, 48, 255, 0.2)"
))

fig10.update_layout(
    xaxis=dict(title="HOG size", tickmode="linear", dtick=1),
    yaxis_title="Number of HOGs",
    width=1000,
    height=500
)

fig10.show()

Data from prefiltered/unexpressed transcripts. These are post tximport in R, so they are transcrtipts that were still mapped by Salmon. The raw orthofinder data includes even more transcripts, but these have no mapping evidence.

In [236]:
salmon_map_unexpressed = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/prefilter_transcripts_annotated.csv", float_precision='legacy')

salmon_map_unexpressed = salmon_map_unexpressed.dropna(subset=["HOG"])
salmon_map_unexpressed
#down from 36382 total transcripts to 30041 without a HOG

,transcript_id,gene_id,seqname,start,end,strand,Description,Preferred_name,HOG,OG,...,COG_category,eggNOG_OGs,age_rank,age_category,duplication_node,gene_tree_node,duplication_support,duplication_type,node_depth_from_root,branch_length
0,g1.t1,g1,utg000001l,185292.0,191963.0,-,-,-,N0.HOG0006038,OG0005418,...,-,"2DQGE@1|root,2S6BS@2759|Eukaryota,3A6U0@33154|...",9.0,recent,C_maculatus_filtered_proteinfasta_TE_filtered,n12,1.0,Terminal,0.736662,0.052941
1,g2.t1,g2,utg000001l,220384.0,220593.0,+,Cytochrome-c oxidase activity,-,N0.HOG0012111,OG0011460,...,I,"2FBM0@1|root,2TCUK@2759|Eukaryota,398E5@33154|...",9.0,recent,C_maculatus_filtered_proteinfasta_TE_filtered,n4,1.0,Terminal,0.736662,0.052941
2,g3.t1,g3,utg000001l,227675.0,244355.0,-,binding. It is involved in the biological proc...,PXK,N0.HOG0004052,OG0003481,...,DUZ,"KOG2101@1|root,KOG2101@2759|Eukaryota,396Y9@33...",9.0,recent,C_maculatus_filtered_proteinfasta_TE_filtered,n15,1.0,Terminal,0.736662,0.052941
3,g4.t1,g4,utg000001l,245866.0,252353.0,+,Hydroxymethylbilane synthase activity. It is i...,HMBS,N0.HOG0004826,OG0004221,...,H,"COG0181@1|root,KOG2892@2759|Eukaryota,38D6W@33...",9.0,recent,C_maculatus_filtered_proteinfasta_TE_filtered,n14,1.0,Terminal,0.736662,0.052941
4,g5.t1,g5,utg000001l,257697.0,260706.0,+,START domain,STARD10,N0.HOG0006068,OG0005448,...,I,"KOG2761@1|root,KOG2761@2759|Eukaryota,38F7R@33...",9.0,recent,C_maculatus_filtered_proteinfasta_TE_filtered,n13,1.0,Terminal,0.736662,0.052941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36377,g35481.t1,g35481,utg003987l,22060.0,24076.0,+,"damaged site, the DNA wraps around one UvrB mo...",uvrB,N0.HOG0012794,OG0012142,...,L,"COG0556@1|root,COG0556@2|Bacteria,1MUFK@1224|P...",9.0,recent,C_maculatus_filtered_proteinfasta_TE_filtered,n3,1.0,Terminal,0.736662,0.052941
36378,g35484.t1,g35484,utg003987l,26411.0,26785.0,+,May be involved in the biosynthesis of molybdo...,moaB,N0.HOG0025288,OG0024635,...,H,"COG0521@1|root,COG0521@2|Bacteria,1R9W2@1224|P...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36379,g35485.t1,g35485,utg003987l,26930.0,27301.0,+,"Catalyzes the conversion of (8S)-3',8-cyclo-7,...",moaC,N0.HOG0025287,OG0024634,...,H,"COG0315@1|root,COG0315@2|Bacteria,1RCYZ@1224|P...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36380,g35486.t1,g35486,utg003987l,27654.0,28118.0,+,PFAM molybdopterin biosynthesis MoaE,moaE,N0.HOG0019449,OG0018796,...,H,"COG0314@1|root,COG0314@2|Bacteria,1RGUX@1224|P...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [237]:
hog_size_unexpressed = (
    salmon_map_unexpressed
    .groupby("HOG")
    .size()
    .reset_index(name="HOG_size")
)

salmon_map_unexpressed = salmon_map_unexpressed.merge(
    hog_size_unexpressed,
    on="HOG",
    how="left"
)

#remove size 1
hog_size_unexpressed = hog_size_unexpressed[hog_size_unexpressed["HOG_size"] >= 2].copy()

hog_size_unexpressed
#We have 14 563 HOGs here. Post DE we had 10,850

,HOG,HOG_size
0,N0.HOG0000001,3
1,N0.HOG0000006,3
2,N0.HOG0000007,58
3,N0.HOG0000008,54
4,N0.HOG0000009,50
...,...,...
14555,N0.HOG0025394,2
14557,N0.HOG0025396,2
14558,N0.HOG0025397,2
14559,N0.HOG0025398,2


Unexpressed density curve plot

In [238]:
from scipy.stats import gaussian_kde

sizes = hog_size_unexpressed["HOG_size"]
x_range = np.linspace(sizes.min(), sizes.max(), 300)

kde = gaussian_kde(sizes)
y_vals = kde(x_range) * len(sizes)

fig11 = go.Figure(go.Scatter(
    x=x_range,
    y=y_vals,
    mode="lines",
    fill="tozeroy",
    line=dict(color="#0030FF", width=2),
    fillcolor="rgba(0, 48, 255, 0.2)"
))

fig11.update_layout(
    xaxis=dict(title="HOG size", tickmode="linear", dtick=10),
    yaxis_title="Number of HOGs",
    width=1000,
    height=500
)

fig11.show()

Overlapping density plot

In [239]:
sizes_pre  = hog_size_unexpressed["HOG_size"]
sizes_post = hog_direction["HOG_size"]

all_sizes_pre  = range(sizes_pre.min(),  sizes_pre.max()  + 1)
all_sizes_post = range(sizes_post.min(), sizes_post.max() + 1)

counts_pre = (
    sizes_pre.value_counts()
    .reindex(all_sizes_pre, fill_value=0)
    .reset_index()
    .set_axis(["HOG_size", "n"], axis=1)
    .sort_values("HOG_size")
)

counts_post = (
    sizes_post.value_counts()
    .reindex(all_sizes_post, fill_value=0)
    .reset_index()
    .set_axis(["HOG_size", "n"], axis=1)
    .sort_values("HOG_size")
)

counts_pre["n_plot"]  = counts_pre["n"].replace(0, 0.1)
counts_post["n_plot"] = counts_post["n"].replace(0, 0.1)


fig11 = go.Figure()

fig11.add_trace(go.Scatter(
    x=counts_pre["HOG_size"],
    y=counts_pre["n"],
    mode="lines",
    name="Pre-filter",
    fill="tozeroy",
    line=dict(color="#0030FF", width=2),
    fillcolor="rgba(0, 48, 255, 0.15)"
))

fig11.add_trace(go.Scatter(
    x=counts_post["HOG_size"],
    y=counts_post["n"],
    mode="lines",
    name="Post-filter",
    fill="tozeroy",
    line=dict(color="#FF0000", width=2),
    fillcolor="rgba(255, 0, 0, 0.15)"
))

fig11.update_layout(
    xaxis=dict(title="HOG size", tickmode="linear", dtick=5),
    yaxis_title="Number of HOGs",
    legend_title_text="Dataset",
    width=1100,
    height=550
)

fig11.show()

Log scaled

In [240]:
fig12 = go.Figure()

fig12.add_trace(go.Scatter(
    x=counts_pre["HOG_size"],
    y=counts_pre["n_plot"],
    customdata=counts_pre["n"],
    mode="lines",
    name="Pre-filter",
    fill="tozeroy",
    line=dict(color="#0030FF", width=2),
    fillcolor="rgba(0, 48, 255, 0.15)",
    hovertemplate="HOG size: %{x}<br>Number of HOGs: %{customdata}<extra></extra>"
))

fig12.add_trace(go.Scatter(
    x=counts_post["HOG_size"],
    y=counts_post["n_plot"],
    customdata=counts_post["n"],
    mode="lines",
    name="Post-filter",
    fill="tozeroy",
    line=dict(color="#FF0000", width=2),
    fillcolor="rgba(255, 0, 0, 0.15)",
    hovertemplate="HOG size: %{x}<br>Number of HOGs: %{customdata}<extra></extra>"
))

fig12.update_layout(
    xaxis=dict(title="HOG size", tickmode="linear", dtick=5),
    yaxis=dict(
        title="Number of HOGs (log scale)",
        type="log",
        tickmode="array",
        tickvals=[1, 10, 100, 1000, 10000],
        ticktext=["1", "10", "100", "1000", "10000"]
    ),
    legend_title_text="Dataset",
    width=1100,
    height=550
)

fig12.show()

# Expressed vs unexpressed mapped transcripts per HOG size 

In [256]:
expressed_ids = set(salmon_map_results_HOG["transcript_id"])

salmon_map_unexpressed["expressed"] = salmon_map_unexpressed["transcript_id"].isin(expressed_ids)
salmon_map_unexpressed["expressed_label"] = salmon_map_unexpressed["expressed"].map(
    {True: "Expressed", False: "Not expressed"}
)

# Filter HOG size >= 2 to match hog_size_unexpressed
expr_counts = (
    salmon_map_unexpressed[salmon_map_unexpressed["HOG_size"] >= 2]
    .groupby(["HOG_size", "expressed_label"])
    .size()
    .reset_index(name="count")
)

size_totals = expr_counts.groupby("HOG_size")["count"].sum().reset_index(name="total")
expr_counts = expr_counts.merge(size_totals, on="HOG_size")
expr_counts["pct"] = expr_counts["count"] / expr_counts["total"] * 100
expr_counts["label"] = expr_counts["count"].where(expr_counts["pct"] >= 4, other="")

fig13 = px.bar(
    expr_counts,
    x="HOG_size",
    y="pct",
    color="expressed_label",
    color_discrete_map={"Expressed": "#0030FF", "Not expressed": "#FF0000"},
    text="label",
    custom_data=["count", "total", "expressed_label"],
    barmode="stack"
)

fig13.update_traces(
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(color="white", size=10),
    hovertemplate=(
        "<b>%{customdata[2]}</b><br>"
        "HOG size: %{x}<br>"
        "Proportion: %{y:.1f}%<br>"
        "Count: %{customdata[0]}<br>"
        "Total at this size: %{customdata[1]}"
        "<extra></extra>"
    )
)

annotations = [
    dict(
        x=row.HOG_size, y=101,
        xref="x", yref="y",
        text=f"<i>n={row.total}</i>",
        showarrow=False,
        font=dict(size=8, color="#444"),
        yanchor="bottom",
        textangle=-45
    )
    for row in size_totals.itertuples()
]

fig13.update_layout(
    xaxis=dict(tickmode="linear", dtick=5, title="HOG size (pre-filter)"),
    yaxis=dict(title="Proportion of transcripts (%)", range=[0, 118], ticksuffix="%"),
    legend_title_text="",
    annotations=annotations,
    margin=dict(b=60, t=60),
    width=1100,
    height=600
)

fig13.update_traces(
    text="",
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(color="white", size=10),
    hovertemplate=(
        "<b>%{customdata[2]}</b><br>"
        "HOG size: %{x}<br>"
        "Proportion: %{y:.1f}%<br>"
        "Count: %{customdata[0]}<br>"
        "Total at this size: %{customdata[1]}"
        "<extra></extra>"
    )
)
fig13.show()

# Plot each transcript vs. HOG size 

In [242]:
#Filter out size 1
salmon_map_results_HOG_filtered = salmon_map_results_HOG[salmon_map_results_HOG["HOG_size"] >= 2].copy()
#Add some jitter on x-axis
np.random.seed(42)  # for reproducibility

jitter = np.random.uniform(
    -0.15, 0.15,
    size=len(salmon_map_results_HOG_filtered)
)

fig10 = px.scatter(
    x=salmon_map_results_HOG_filtered["HOG_size"] + jitter,
    y=salmon_map_results_HOG_filtered["log2FoldChange"],
    opacity=0.4,
    labels={
        "x": "HOG size (number of paralogs)",
        "y": "Transcript log2FoldChange"
    },
    title="Transcript-level sex bias as a function of HOG size"
)

fig10.add_hline(y=0, line_dash="dash")
fig10.show()


# Average HOG sex bias

In [243]:
#Do absolute instead of directional? 
hog_lfc = (
    salmon_map_results_HOG
    .groupby("HOG")["log2FoldChange"]
    .mean()
    .reset_index(name="mean_log2FC")
)

Merge

In [244]:
hog_summary = hog_lfc.merge(hog_size, on="HOG")
hog_summary

,HOG,mean_log2FC,HOG_size
0,N0.HOG0000007,1.846158,17
1,N0.HOG0000008,1.672112,17
2,N0.HOG0000009,2.084351,14
3,N0.HOG0000010,1.060875,25
4,N0.HOG0000011,-1.410419,4
...,...,...,...
10845,N0.HOG0024543,0.434326,2
10846,N0.HOG0024547,-0.138706,1
10847,N0.HOG0024667,0.431166,1
10848,N0.HOG0024673,2.382906,1


Plot

In [245]:
#Filter out size 1
hog_summary_filtered = hog_summary[hog_summary["HOG_size"] >= 2].copy()

#Add some jitter on x-axis
np.random.seed(42)  # for reproducibility

hog_summary_filtered["HOG_size_jitter"] = (
    hog_summary_filtered["HOG_size"]
    + np.random.uniform(-0.15, 0.15, size=len(hog_summary_filtered))
)


fig11 = px.scatter(
    hog_summary_filtered,
    x="HOG_size_jitter",
    y="mean_log2FC",
    hover_name="HOG",
    opacity=0.6,
    labels={
        "HOG_size_jitter": "HOG size (number of paralogs)",
        "mean_log2FC": "Mean log2FoldChange"
    },
    title="Paralog number vs sex-biased expression (Directional)"
)

fig11.add_hline(y=0, line_dash="dash")
fig11.show()


Try with absolute L2FC

In [246]:
hog_lfc_abs = (
    salmon_map_results_HOG
    .assign(abs_log2FC=lambda x: x["log2FoldChange"].abs())
    .groupby("HOG")["abs_log2FC"]
    .mean()
    .reset_index(name="mean_abs_log2FC")
)

hog_abs_summary = hog_lfc_abs.merge(hog_size, on="HOG")


In [247]:
hog_abs_summary_filtered = hog_abs_summary[hog_abs_summary["HOG_size"] >= 2].copy()

#Add some jitter on x-axis
np.random.seed(42)  # for reproducibility

hog_abs_summary_filtered["HOG_size_jitter"] = (
    hog_abs_summary_filtered["HOG_size"]
    + np.random.uniform(-0.15, 0.15, size=len(hog_abs_summary_filtered))
)

fig12 = px.scatter(
    hog_abs_summary_filtered,
    x="HOG_size_jitter",
    y="mean_abs_log2FC",
    hover_name="HOG",
    opacity=0.6,
    labels={
        "HOG_size_jitter": "HOG size (number of paralogs)",
        "mean_abs_log2FC": "Mean |log2FoldChange|"
    },
    title="Paralog number vs strength of sex-biased expression (Absolute)"
)

fig12.show()


# Male-bias only and female-bias only

In [248]:
#split each sex up based on log2fc
male_biased = salmon_map_results_HOG[
    salmon_map_results_HOG["log2FoldChange"] > 0
]

female_biased = salmon_map_results_HOG[
    salmon_map_results_HOG["log2FoldChange"] < 0
]


In [249]:
# male-biased
hog_male = (
    male_biased
    .groupby("HOG")["log2FoldChange"]
    .mean()
    .reset_index(name="mean_male_log2FC")
)


In [250]:
#female biased
hog_female = (
    female_biased
    .assign(abs_log2FC=lambda x: x["log2FoldChange"].abs())
    .groupby("HOG")["abs_log2FC"]
    .mean()
    .reset_index(name="mean_female_log2FC")
)


In [251]:
#hog size same as before
hog_size = (
    salmon_map_results_HOG
    .groupby("HOG")
    .size()
    .reset_index(name="HOG_size")
)


In [252]:
#merge all
hog_sex_bias = (
    hog_size
    .merge(hog_male, on="HOG", how="left")
    .merge(hog_female, on="HOG", how="left")
)


hog_sex_bias


,HOG,HOG_size,mean_male_log2FC,mean_female_log2FC
0,N0.HOG0000007,17,1.846158,NaN
1,N0.HOG0000008,17,1.802686,0.417072
2,N0.HOG0000009,14,2.263200,0.240689
3,N0.HOG0000010,25,1.126954,0.524999
4,N0.HOG0000011,4,NaN,1.410419
...,...,...,...,...
10845,N0.HOG0024543,2,0.434326,NaN
10846,N0.HOG0024547,1,NaN,0.138706
10847,N0.HOG0024667,1,0.431166,NaN
10848,N0.HOG0024673,1,2.382906,NaN


plot male biased transcripts

In [253]:
#Add some jitter on x-axis
np.random.seed(42)  # for reproducibility

hog_sex_bias["HOG_size_jitter"] = (
    hog_sex_bias["HOG_size"]
    + np.random.uniform(-0.15, 0.15, size=len(hog_sex_bias))
)

fig = px.scatter(
    hog_sex_bias,
    x="HOG_size_jitter",
    y="mean_male_log2FC",
    opacity=0.6,
    labels={
        "HOG_size_jitter": "HOG size",
        "mean_male_log2FC": "Mean male-biased log2FC"
    },
    title="Paralog number vs male-biased expression strength"
)
fig.show()


Plot female baised transcripts

In [254]:
fig = px.scatter(
    hog_sex_bias,
    x="HOG_size_jitter",
    y="mean_female_log2FC",
    opacity=0.6,
    labels={
        "HOG_size_jitter": "HOG size",
        "mean_female_log2FC": "Mean female-biased |log2FC|"
    },
    title="Paralog number vs female-biased expression strength"
)
fig.show()
